In [ ]:
# Import modules
import pandas as pd
import numpy as np
from plotly.offline import init_notebook_mode
from IPython.display import display, HTML

# Allows you to use modified modules without rebooting the kernel
%load_ext autoreload
%autoreload 2
# Enable latex on plotly figures
init_notebook_mode()
display(
    HTML(
        '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
    )
)

# Data that would be used for testing

In [ ]:
magData = pd.read_csv("../../src/datasets/2020_ydhm_id.csv")
magData.index = pd.DatetimeIndex(magData["ydhm_id"])
magData = magData.drop(columns=["ydhm_id"])
# Consider only one month - January
# magData = magData[magData.index < pd.to_datetime("2020-02-01 00:00:00")]
N = magData.shape[0]
dataset = magData[magData.index < pd.to_datetime("2020-02-01 00:00:00")]
# Free space
del magData
# Show dataset
dataset.diff().head(10)

In [ ]:
import tensorflow_probability
from magfield.em.mixture import Mixture

# Create mixture model
m = Mixture(num_comps=3, distrib=tensorflow_probability.distributions.Normal)
np.random.seed()
rseed = np.random.randint(1_000)
m.initialize_probs_mus_sigmas(random_seed=rseed)
print(f"radnom seed equal to {rseed}")
# Generate data for testing
m.generate_samples(7_000, random_seed=57)
# another_data = m.construct_tpf_mixture().sample(1_000).numpy()
test_data1 = m.samples
# test_data2 = another_data


## additional stuff; output appearancece
def __round(numb, dig=4):
    return round(numb, dig)


def __Round(arr, dig=4):
    return list(map(lambda numb: round(numb, dig), arr))


def custom_print(text, *args, func=__Round):
    print(
        text,
        f"Orig - {func(args[0])}",
        f"Iter - {func(args[1])}",
        f"Adap - {func(args[2])}",
        sep="\n\t",
    )


# EM comparison
pit, mit, sit, llhit = m.EM_iterative(test_data1, 30)
pad, mad, sad, llhad = m.EM_adaptive(test_data1, 0.001)

custom_print("Components probabilities:", m.probs, pit, pad)
custom_print("Mathematical expectations:", m.mus, mit, mad)
custom_print("Standard deviations:", m.sigmas, sit, sad)
custom_print(
    "Log-likelihoods:", m.log_likelihood(test_data1), llhit, llhad, func=__round
)

In [ ]:
import tensorflow_probability
from magfield.em.mixture import Mixture

# Create mixture model
m = Mixture(num_comps=3, distrib=tensorflow_probability.distributions.Normal)
np.random.seed()
rseed = np.random.randint(4_000)
m.initialize_probs_mus_sigmas(random_seed=rseed)
print(f"radnom seed equal to {rseed}")
# Generate data for testing
m.generate_samples(1_000, random_seed=57)
# another_data = m.construct_tpf_mixture().sample(1_000).numpy()
test_data1 = m.samples

# EM apply

In [ ]:
from magfield.EM import EMiter, EMadap, EMsiev, EMKS

iterative = EMiter(num_comp=3, num_iter=100)
iterative.fit(test_data1)
print("Original parameters:", *m.parameters.items(), sep="\n")
print(iterative)

In [ ]:
adaptive = EMadap(num_comp=3, epsilon=0.001)
adaptive.fit(test_data1)
print("Original parameters:", *m.parameters.items(), sep="\n")
print(adaptive)

In [ ]:
sieving = EMsiev(num_comp=3, num_init=10, num_iter=100, epsilon=0.001)
sieving.fit(test_data1)
print("Original parameters:", *m.parameters.items(), sep="\n")
print("Sieving parameters:\n", sieving)

In [ ]:
kolmogorov = EMKS(num_comp=3)
kolmogorov.fit(test_data1, train_perc=0.8, conv_speed=0.0001)
print("Original parameters:", *m.parameters.items(), sep="\n")
print("Kolmogorov parameters:\n", kolmogorov)

In [ ]:
print(iterative._llh, adaptive._llh, sieving._llh, kolmogorov._llh, sep="\n")
print(iterative.aic, adaptive.aic, sieving.aic, kolmogorov.aic, sep="\n")